# Stage 1 — Case Base: Data Acquisition & Preprocessing

**Project:** Case-Based Reasoning (CBR) untuk Analisis Putusan Pengadilan Indonesia  
**Tahap:** 1 dari 4 — Membangun Case Base  
**Environment:** VSCode + Jupyter (Lokal)  

---

## Overview

Notebook ini membangun **case base** untuk sistem CBR dengan cara:

1. Memuat file PDF putusan pengadilan dari folder dataset lokal.
2. Melakukan extraction teks polos menggunakan `pdfplumber`.
3. Melakukan cleaning pada teks hasil extraction (watermark, whitespace, format teks).
4. Menyimpan hasil cleaning teks ke dalam struktur folder project.
5. Menghasilkan log preprocessing dan laporan validation.

**Keputusan Desain Penting:**
- Tanpa stemming, lemmatization, atau stopword removal — teks hukum memerlukan preservasi konteks yang utuh.
- Pembersihan watermark yang disesuaikan dengan format PDF Mahkamah Agung.
- Pipeline dapat diskalakan untuk 30+ dokumen tanpa perubahan kode.

---

## Daftar Isi

1. [Environment Setup](#1.-Environment-Setup)
2. [Configure Paths](#2.-Configure-Paths)
3. [Create Project Structure](#3.-Create-Project-Structure)
4. [Scan PDF Files](#4.-Scan-PDF-Files)
5. [PDF Text Extraction](#5.-PDF-Text-Extraction)
6. [Text Cleaning](#6.-Text-Cleaning)
7. [Save Cleaned Text](#7.-Save-Cleaned-Text)
8. [Logging](#8.-Logging)
9. [Validation](#9.-Validation)
10. [Final Summary](#10.-Final-Summary)

**Lampiran (Appendices):**
- [A. Future Preprocessing Improvements](#A.-Future-Preprocessing-Improvements)
- [B. Common Pitfalls dengan PDF Hukum](#B.-Common-Pitfalls-dengan-PDF-Hukum)
- [C. Indonesian Legal Text Best Practices](#C.-Indonesian-Legal-Text-Best-Practices)
- [D. Preprocessing Validation Checklist](#D.-Preprocessing-Validation-Checklist)

---

# 1. Environment Setup

Instalasi dependencies yang diperlukan dan import seluruh library untuk pipeline.

**Library yang digunakan:**
- `pdfplumber` — PDF text extraction (layout-aware, tanpa OCR).
- `pandas` — Manipulasi data dan pembuatan tabel summary.
- `tqdm` — Progress bar untuk batch processing.
- `pathlib`, `os`, `glob`, `re` — File handling dan pemrosesan teks (stdlib).

In [1]:
# Install dependencies (run once per environment)

%pip install pdfplumber tqdm pandas -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import libraries

import os
import re
import glob
import logging
import warnings
from pathlib import Path
from datetime import datetime

import pdfplumber
import pandas as pd
from tqdm import tqdm

# Suppress noisy warnings from pdfplumber internals
warnings.filterwarnings('ignore')

print("All libraries imported successfully")
print(f"Session started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

All libraries imported successfully
Session started: 2026-06-24 21:14:40


---

# 2. Configure Paths

Mengatur path project root dan dataset pada filesystem lokal.

Path project root akan di-resolve secara otomatis berdasarkan lokasi notebook ini.  
Folder dataset (`datasets uu-ite/`) berisi file PDF putusan pengadilan.

In [3]:
from pathlib import Path
import os

# CONFIGURATION — Local path setup

PROJECT_ROOT = Path("../").resolve()

DATASET_DIR = PROJECT_ROOT / "datasets" / "uu-ite"

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
LOG_DIR = PROJECT_ROOT / "logs"

# Verify paths


print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset folder: {DATASET_DIR}")
print()

if DATASET_DIR.exists():

    contents = os.listdir(DATASET_DIR)

    pdf_count = sum(
        1 for f in contents
        if f.lower().endswith(".pdf")
    )

    print(" Dataset folder found")
    print(f"   Total items: {len(contents)}")
    print(f"   PDF files: {pdf_count}")

else:

    print(f"❌ Dataset folder NOT found:")
    print(DATASET_DIR)

Project root: D:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)
Dataset folder: D:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\datasets\uu-ite

 Dataset folder found
   Total items: 44
   PDF files: 44


---

# 3. Create Project Structure

Membuat seluruh direktori project yang diperlukan secara otomatis.

```text
project-cbr/
├── datasets uu-ite/  # PDF putusan pengadilan (input)
├── data/
│   ├── raw/          # File teks bersih (output Stage 1)
│   ├── processed/    # Representasi kasus terstruktur (Stage 2)
│   ├── eval/         # Dataset evaluasi (Stage 4)
│   └── results/      # Summary pipeline dan laporan
├── notebooks/        # Jupyter notebook
└── logs/             # Log pemrosesan
```

In [4]:
# Create project directory structure

DIRECTORIES = [
    PROJECT_ROOT / 'data' / 'raw',
    PROJECT_ROOT / 'data' / 'processed',
    PROJECT_ROOT / 'data' / 'eval',
    PROJECT_ROOT / 'data' / 'results',
    PROJECT_ROOT / 'notebooks',
    PROJECT_ROOT / 'logs',
]

print("📁 Creating project structure...\n")

for dir_path in DIRECTORIES:
    dir_path.mkdir(parents=True, exist_ok=True)
    # Show relative path for readability
    relative = dir_path.relative_to(PROJECT_ROOT)
    print(f"  📂 {relative}/")

print(f"\n✅ Project structure created at: {PROJECT_ROOT}")

📁 Creating project structure...

  📂 data\raw/
  📂 data\processed/
  📂 data\eval/
  📂 data\results/
  📂 notebooks/
  📂 logs/

✅ Project structure created at: D:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)


---

# 4. Scan PDF Files

Mendeteksi secara otomatis seluruh file PDF di dalam folder dataset lokal.  
Tidak ada nama file yang di-hardcode — pipeline secara dinamis menemukan seluruh file `.pdf`.

Ini mendukung skalabilitas untuk 30+ dokumen tanpa adanya perubahan kode.

In [5]:
# Auto-scan all PDF files in the dataset folder

def scan_pdf_files(folder_path):
    """
    Scan a folder for PDF files (case-insensitive extension matching).

    Args:
        folder_path (str): Path to the folder to scan.

    Returns:
        list[str]: Sorted list of absolute paths to PDF files.
    """
    pdf_files = []
    for ext in ('*.pdf', '*.PDF'):
        pdf_files.extend(glob.glob(os.path.join(folder_path, ext)))

    # Remove duplicates (in case filesystem is case-insensitive) and sort
    pdf_files = sorted(set(pdf_files))
    return pdf_files


# Run the scan
pdf_files = scan_pdf_files(DATASET_DIR)

if pdf_files:
    print(f"📄 Found {len(pdf_files)} PDF file(s):\n")
    for i, pdf_path in enumerate(pdf_files, 1):
        file_size_kb = os.path.getsize(pdf_path) / 1024
        print(f"  {i:3d}. {os.path.basename(pdf_path):50s} ({file_size_kb:,.1f} KB)")
else:
    print(f"No PDF files found in: {DATASET_DIR}")
    print("   Please check the folder path and try again.")

📄 Found 44 PDF file(s):

    1. putusan_107_pid.sus_2021_pn_jkt.sel_20260624185710.pdf (255.1 KB)
    2. putusan_10_pid.sus_2025_pt_bdg_20260624190424.pdf  (104.0 KB)
    3. putusan_1174_pid.sus_2021_pn_jkt.utr_20260624185123.pdf (147.5 KB)
    4. putusan_1221_pid.sus_2020_pt_sby_20260624184940.pdf (127.1 KB)
    5. putusan_134_pid.sus_2018_pn_bnr_20260624185857.pdf (248.4 KB)
    6. putusan_1357_pid.sus_2021_pt_mdn_20260624190325.pdf (214.0 KB)
    7. putusan_1536_pid.sus_2019_pn_jkt.brt_20260624185350.pdf (285.4 KB)
    8. putusan_1537_pid.sus_2019_pn_jkt.brt_20260624185405.pdf (304.4 KB)
    9. putusan_1597_pid.sus_2019_pn_jkt.utr_20260624185744.pdf (173.5 KB)
   10. putusan_159_pid.sus_2023_pn_sda_20260624190433.pdf (476.4 KB)
   11. putusan_165_pid.sus_2020_pn_jkt.utr_20260624124027.pdf (192.1 KB)
   12. putusan_167_pid.sus_2019_pn_ckr_20260624185039.pdf (164.3 KB)
   13. putusan_1682_pid.sus_2025_pn_tng_20260624190003.pdf (168.9 KB)
   14. putusan_2584_pid.sus_2025_pn_sby_2026062

---

# 5. PDF Text Extraction

Mengekstrak teks dari setiap PDF menggunakan **pdfplumber**, yang menyediakan layout-aware text extraction tanpa memerlukan proses OCR.

**Mengapa pdfplumber?**
- Menangani tata letak multi-kolom yang umum ditemukan pada putusan pengadilan Indonesia.
- Mempertahankan urutan pembacaan teks dengan lebih baik dibanding `pdfminer` biasa.
- Tidak memiliki dependency eksternal (Tesseract, Poppler, dll.).

**Batasan (Limitations):**
- Tidak dapat mengekstrak teks dari PDF hasil scan/hanya gambar (memerlukan OCR fallback).
- Beberapa PDF dengan pemformatan kompleks mungkin menghasilkan output teks yang berantakan.

In [6]:
# Define the PDF text extraction function

def extract_text_from_pdf(pdf_path):
    """
    Extract all text from a PDF file using pdfplumber.

    Iterates over every page, extracts text, and joins them
    with double-newlines as page separators.

    Args:
        pdf_path (str): Absolute path to the PDF file.

    Returns:
        dict: Extraction result with keys:
            - filename (str): Original PDF filename
            - filepath (str): Full path to the PDF
            - raw_text (str): Extracted raw text
            - page_count (int): Number of pages in the PDF
            - status (str): 'success', 'warning', or 'failed'
            - error (str|None): Error message if any
    """
    result = {
        'filename': os.path.basename(pdf_path),
        'filepath': pdf_path,
        'raw_text': '',
        'page_count': 0,
        'status': 'success',
        'error': None,
    }

    try:
        with pdfplumber.open(pdf_path) as pdf:
            result['page_count'] = len(pdf.pages)
            pages_text = []

            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    pages_text.append(text)

            result['raw_text'] = '\n\n'.join(pages_text)

            # Flag if no text was extracted (likely a scanned PDF)
            if not result['raw_text'].strip():
                result['status'] = 'warning'
                result['error'] = 'No text extracted — possible scanned/image-only PDF'

    except Exception as e:
        result['status'] = 'failed'
        result['error'] = str(e)

    return result

In [7]:
# Run extraction on all detected PDFs

print(" Extracting text from PDF files...\n")

extraction_results = []

for pdf_path in tqdm(pdf_files, desc="Extracting PDFs"):
    result = extract_text_from_pdf(pdf_path)
    extraction_results.append(result)

    # Status indicator
    icons = {'success': '✅', 'warning': '⚠️', 'failed': '❌'}
    icon = icons.get(result['status'], '❓')
    char_count = len(result['raw_text'])

    print(f"  {icon} {result['filename']}")
    print(f"     Pages: {result['page_count']}, Characters: {char_count:,}")
    if result['error']:
        print(f"     ⚠ {result['error']}")

# Quick tally
n_ok = sum(1 for r in extraction_results if r['status'] == 'success')
n_warn = sum(1 for r in extraction_results if r['status'] == 'warning')
n_fail = sum(1 for r in extraction_results if r['status'] == 'failed')

print(f"\n Extraction complete — {n_ok} success, {n_warn} warning(s), {n_fail} failure(s)")

 Extracting text from PDF files...



Extracting PDFs:   0%|          | 0/44 [00:00<?, ?it/s]

Extracting PDFs:   2%|▏         | 1/44 [00:05<03:37,  5.07s/it]

  ✅ putusan_107_pid.sus_2021_pn_jkt.sel_20260624185710.pdf
     Pages: 41, Characters: 140,425


Extracting PDFs:   5%|▍         | 2/44 [00:05<01:49,  2.61s/it]

...

  ✅ putusan_88_pid.sus_2025_pn_rbg_20260624190017.pdf
     Pages: 36, Characters: 120,795


Extracting PDFs:  98%|█████████▊| 43/44 [03:04<00:04,  4.66s/it]

  ✅ putusan_980_pid.sus_2024_pn_bdg_20260624185133.pdf
     Pages: 39, Characters: 132,858


Extracting PDFs: 100%|██████████| 44/44 [03:12<00:00,  4.37s/it]

  ✅ putusan_9_pid.sus_2026_pn_ngb_20260623014917.pdf
     Pages: 68, Characters: 230,345

 Extraction complete — 44 success, 0 warning(s), 0 failure(s)


---

# 6. Text Cleaning

Clean teks hasil extraction dengan tetap menjaga makna hukum di dalamnya.

**Tahap cleaning yang diterapkan:**
1. Menghapus watermark yang diketahui (disclaimer Mahkamah Agung, URL, marker halaman) — regex block-level.
2. Mengubah teks menjadi lowercase.
3. Menghapus karakter non-printable / control characters.
4. Menormalisasi horizontal whitespace (tab, multiple spaces menjadi single space).
5. Menghilangkan trailing whitespace pada setiap baris teks.
6. Menghapus baris disclaimer/watermark yang tersisa (substring-based, lebih robust terhadap garbling) dan fragmen watermark vertikal (`ma ubl`).
7. Menghapus karakter noise tunggal di awal/akhir baris yang bocor dari watermark sidebar vertikal (dengan safeguard untuk referensi hukum seperti "huruf a" dan kata berspasi seperti "p u t u s a n").
8. Mereduksi baris kosong berlebih (3+ baris kosong berturut-turut menjadi 2 baris kosong).

**Mengapa digabung jadi satu fungsi?**
Watermark pada dokumen putusan Mahkamah Agung muncul dalam dua bentuk: blok disclaimer multi-baris yang bisa ditangkap regex, dan fragmen/noise per-baris yang baru terlihat *setelah* lowercasing & whitespace normalization. Karena itu deteksi substring & noise inline dijalankan sebagai langkah lanjutan dalam fungsi `clean_text` yang sama, bukan sebagai notebook/proses terpisah — sehingga seluruh pembersihan teks terjadi dalam satu pass yang konsisten.

**Sengaja TIDAK diterapkan:**
- ❌ Stemming — karena dapat merusak terminologi hukum (misal: "memutuskan" vs "putusan").
- ❌ Lemmatization — alasan yang sama.
- ❌ Stopword removal — kata seperti "dan", "atau", "tidak" memiliki interpretasi hukum yang sangat penting.


In [8]:
# Define watermark patterns and the text cleaning function

# Common watermarks/headers/footers found in Indonesian court decision PDFs
# (particularly those downloaded from putusan.mahkamahagung.go.id)
WATERMARK_PATTERNS = [
    # Mahkamah Agung directory header
    r'Direktori\s+Putusan\s+Mahkamah\s+Agung\s+Republik\s+Indonesia',
    # Website watermark
    r'putusan\.mahkamahagung\.go\.id',
    # Standard disclaimer block (multi-line, captured generously)
    r'Disclaimer\s*\n.*?Kepaniteraan\s+Mahkamah\s+Agung.*?(?=\n\n|\Z)',
    # Page number markers: "Halaman X dari Y ..."
    r'Halaman\s+\d+\s+dari\s+\d+[^\n]*',
    # Sometimes the full court header repeats on every page
    r'Mahkamah\s+Agung\s+Republik\s+Indonesia\s*\n\s*Mahkamah\s+Agung\s+Republik\s+Indonesia',
]

# Substring markers — sisa watermark/disclaimer yang mengalami garbling sehingga
# tidak tertangkap oleh regex block-level di atas. Jika SALAH SATU substring ini
# ada di sebuah baris (lowercase), baris tersebut adalah artefak disclaimer/watermark.
WATERMARK_MARKERS = [
    "selalu mencantumkan informasi",
    "paling kini dan akurat",
    "pelayanan publik, transparansi",
    "gsi peradilan",
    "menemukan inakurasi",
    "mahkamahagung.go.id",
    "021-384 3348",
]

# Baris yang persis sama dengan fragmen watermark vertikal (setelah strip + lowercase)
WATERMARK_EXACT = {"ma ubl", "ma ubl."}

# Deteksi inline noise: karakter tunggal di awal/akhir baris (bocor dari watermark
# sidebar vertikal "Mahkamah Agung Republik Indonesia").
LEADING_NOISE_RE = re.compile(r"^([a-z]) (?=[a-z]{2,}|\d)")
TRAILING_NOISE_RE = re.compile(r"\s+[a-z]$")

# Kata referensi hukum yang sah diikuti huruf tunggal (misal: "huruf a") — jangan dihapus.
LEGAL_REF_WORDS = {"huruf", "sub", "poin", "butir", "angka", "lampiran", "golongan"}


def is_watermark_line(line_text):
    """Deteksi apakah sebuah baris adalah artefak disclaimer/watermark (substring-based)."""
    stripped = line_text.strip().lower()
    if not stripped:
        return False
    if stripped in WATERMARK_EXACT:
        return True
    return any(marker in stripped for marker in WATERMARK_MARKERS)


# Vertical sidebar watermark sering bocor sebagai SATU baris per karakter
# (contoh: baris-baris berisi "a", "i", "s", "e", "n", "o", "d", "n", "i" yang
# saat dibaca terbalik membentuk "indonesia"). Baris semacam ini hampir tidak
# pernah merupakan isi hukum yang sah, sehingga dapat dihapus seluruhnya.
SINGLE_CHAR_LINE_RE = re.compile(r"^[a-z]$")


def is_single_char_bleed(line_text):
    """Deteksi baris yang HANYA berisi satu karakter alfabet (vertical watermark bleed)."""
    return bool(SINGLE_CHAR_LINE_RE.match(line_text.strip()))


def has_spaced_word_suffix(text):
    """Cek apakah baris diakhiri kata berspasi seperti 'p u t u s a n' (bukan noise)."""
    return bool(re.search(r"(?:[a-z] ){2,}[a-z]$", text.strip()))


def _strip_inline_noise(line):
    """Hapus karakter noise tunggal di awal/akhir baris, dengan safeguard hukum."""
    stripped = LEADING_NOISE_RE.sub('', line)

    m = TRAILING_NOISE_RE.search(stripped)
    if m and not has_spaced_word_suffix(stripped):
        words = stripped.rsplit(maxsplit=2)
        if not (len(words) >= 2 and words[-2].lower() in LEGAL_REF_WORDS):
            stripped = TRAILING_NOISE_RE.sub('', stripped)

    return stripped


def clean_text(text):
    """
    Clean extracted text from an Indonesian court decision PDF.

    Applies surface-level cleaning to normalize the text while
    preserving all legal content and context.

    Steps:
    1. Remove known block-level watermarks/disclaimers (regex).
    2. Convert to lowercase.
    3. Remove non-printable/control characters.
    4. Normalize horizontal whitespace.
    5. Strip leading/trailing whitespace on each line.
    6. Remove remaining disclaimer/watermark lines (substring-based) and
       vertical watermark fragments (e.g. "ma ubl").
    7. Remove single-character lines bled from the vertical sidebar watermark
       (e.g. lines reading just "a", "i", "s", "e", "n", "o", "d", "n", "i").
    8. Strip leading/trailing single-char noise leaked from vertical watermark.
    9. Collapse excessive blank lines.

    Args:
        text (str): Raw extracted text from a PDF.

    Returns:
        str: Cleaned text.
    """
    if not text or not text.strip():
        return ''

    cleaned = text

    # Step 1: Remove known block-level watermarks, headers, and footers
    for pattern in WATERMARK_PATTERNS:
        cleaned = re.sub(pattern, '', cleaned, flags=re.IGNORECASE | re.DOTALL)

    # Step 2: Convert to lowercase
    cleaned = cleaned.lower()

    # Step 3: Remove non-printable/control characters
    # Keep: printable ASCII, standard whitespace, and extended Latin characters
    # (covers Indonesian text which uses standard Latin alphabet)
    cleaned = re.sub(r'[^\x20-\x7E\n\r\t\u00C0-\u024F]', '', cleaned)

    # Step 4: Normalize horizontal whitespace (tabs + multiple spaces -> single space)
    cleaned = re.sub(r'[^\S\n]+', ' ', cleaned)

    # Step 5: Strip leading/trailing whitespace on each line
    lines = [line.strip() for line in cleaned.split('\n')]

    # Step 6-7: Remove residual watermark lines + strip inline noise, line by line
    fixed_lines = []
    for line in lines:
        if not line:
            fixed_lines.append('')
            continue
        if is_watermark_line(line):
            continue
        if is_single_char_bleed(line):
            continue
        fixed_lines.append(_strip_inline_noise(line))

    cleaned = '\n'.join(fixed_lines)

    # Step 8: Collapse 3+ consecutive blank lines into a single blank line
    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned)

    # Final strip
    cleaned = cleaned.strip()

    return cleaned


print("✅ Cleaning function defined")
print(f"   {len(WATERMARK_PATTERNS)} block-level watermark pattern(s) loaded")
print(f"   {len(WATERMARK_MARKERS)} substring watermark marker(s) loaded")
print(f"   single-character vertical bleed lines will be removed")


✅ Cleaning function defined
   5 block-level watermark pattern(s) loaded
   7 substring watermark marker(s) loaded
   single-character vertical bleed lines will be removed


In [9]:
# Apply cleaning to all extracted texts

print(" Cleaning extracted texts...\n")

for result in tqdm(extraction_results, desc="Cleaning"):
    if result['status'] in ('success', 'warning') and result['raw_text']:
        result['cleaned_text'] = clean_text(result['raw_text'])
        result['raw_char_count'] = len(result['raw_text'])
        result['cleaned_char_count'] = len(result['cleaned_text'])

        # Calculate size reduction
        if result['raw_char_count'] > 0:
            reduction_pct = (1 - result['cleaned_char_count'] / result['raw_char_count']) * 100
        else:
            reduction_pct = 0.0

        print(f"  [DONE] {result['filename']}")
        print(f"     {result['raw_char_count']:,} → {result['cleaned_char_count']:,} chars "
              f"({reduction_pct:.1f}% reduction)")
    else:
        result['cleaned_text'] = ''
        result['raw_char_count'] = 0
        result['cleaned_char_count'] = 0
        print(f"{result['filename']}: skipped (no text available)")

print(f"\n Cleaning complete for {len(extraction_results)} file(s)")

 Cleaning extracted texts...



Cleaning:   0%|          | 0/44 [00:00<?, ?it/s]

  [DONE] putusan_107_pid.sus_2021_pn_jkt.sel_20260624185710.pdf
     140,425 → 91,347 chars (34.9% reduction)
  [DONE] putusan_10_pid.sus_2025_pt_bdg_20260624190424.pdf
     27,462 → 16,671 chars (39.3% reduction)
  [DONE] putusan_1174_pid.sus_2021_pn_jkt.utr_20260624185123.pdf
     50,333 → 31,192 chars (38.0% reduction)
  [DONE] putusan_1221_pid.sus_2020_pt_sby_20260624184940.pdf
     56,034 → 32,505 chars (42.0% reduction)
  [DONE] putusan_134_pid.sus_2018_pn_bnr_20260624185857.pdf
     129,282 → 84,810 chars (34.4% reduction)
  [DONE] putusan_1357_pid.sus_2021_pt_mdn_20260624190325.pdf
     127,986 → 85,433 chars (33.2% reduction)


Cleaning:  14%|█▎        | 6/44 [00:00<00:00, 50.73it/s]

  [DONE] putusan_1536_pid.sus_2019_pn_jkt.brt_20260624185350.pdf
     170,996 → 112,755 chars (34.1% reduction)
  [DONE] putusan_1537_pid.sus_2019_pn_jkt.brt_20260624185405.pdf
     177,648 → 114,910 chars (35.3% reduction)
  [DONE] putusan_1597_pid.sus_2019_pn_jkt.utr_20260624185744.pdf
     84,502 → 53,379 chars (36.8% reduction)
  [DONE] putusan_159_pid.sus_2023_pn_sda_20260624190433.pdf
     263,174 → 171,141 chars (35.0% reduction)
  [DONE] putusan_165_pid.sus_2020_pn_jkt.utr_20260624124027.pdf
     92,094 → 59,818 chars (35.0% reduction)
  [DONE] putusan_167_pid.sus_2019_pn_ckr_20260624185039.pdf
     58,983 → 37,426 chars (36.5% reduction)


...

  [DONE] putusan_2831_pid.sus_2025_pn_sby_20260624134018.pdf
     46,447 → 29,641 chars (36.2% reduction)
  [DONE] putusan_32_pid.sus_2020_pn_mjn_20260624185654.pdf
     75,980 → 47,309 chars (37.7% reduction)
  [DONE] putusan_331_pid.sus_2019_pt_dki_20260624190333.pdf
     71,814 → 47,884 chars (33.3% reduction)
  [DONE] putusan_337_pid.sus_2020_pt_dki_20260624124151.pdf
     34,669 → 22,167 chars (36.1% reduction)
  [DONE] putusan_3399_pid.sus_2019_pn_sby_20260624112132.pdf
     49,616 → 31,695 chars (36.1% reduction)
  [DONE] putusan_371_pid.sus_2025_pn_gpr_20260624190459.pdf
     98,662 → 65,283 chars (33.8% reduction)
  [DONE] putusan_394_pid.sus_2018_pn_pbr_20260624095617.pdf
     193,726 → 116,766 chars (39.7% reduction)
  [DONE] putusan_3_pid.sus_2020_pn_sgr_20260624185521.pdf
     62,558 → 39,909 chars (36.2% reduction)
  [DONE] putusan_419_pid.sus_2023_pn_jkt.sel_20260623174821.pdf
     117,325 → 70,327 chars (40.1% reduction)


Cleaning:  80%|███████▉  | 35/44 [00:00<00:00, 63.83it/s]

  [DONE] putusan_446_pid.sus_2021_pn_jkt.utr_20260624190610.pdf
     112,694 → 70,631 chars (37.3% reduction)
  [DONE] putusan_451_pid.sus_2025_pn_dps_20260624185850.pdf
     96,923 → 61,152 chars (36.9% reduction)
  [DONE] putusan_52_pid.sus_2021_pn_dgl_20260624190339.pdf
     77,360 → 48,774 chars (37.0% reduction)
  [DONE] putusan_582_pid.sus_2025_pn_jkt.sel_20260624185830.pdf
     97,342 → 61,255 chars (37.1% reduction)
  [DONE] putusan_616_pid.sus_2021_pn_jkt.sel_20260624185228.pdf
     102,037 → 64,818 chars (36.5% reduction)
  [DONE] putusan_617_pid.sus_2021_pn_jkt.sel_20260624185142.pdf
     93,978 → 59,192 chars (37.0% reduction)
  [DONE] putusan_63_pid.sus_2025_pn_ngb_20260624190512.pdf
     153,992 → 101,551 chars (34.1% reduction)
  [DONE] putusan_658_pid.sus_2021_pt_mks_20260624185423.pdf
     51,761 → 32,577 chars (37.1% reduction)
  [DONE] putusan_68_pid.sus_2022_pn_pal_20260624185049.pdf
     87,796 → 55,405 chars (36.9% reduction)
  [DONE] putusan_724_pid.sus_2024_pn_d

Cleaning: 100%|██████████| 44/44 [00:00<00:00, 56.10it/s]

  [DONE] putusan_88_pid.sus_2025_pn_rbg_20260624190017.pdf
     120,795 → 77,855 chars (35.5% reduction)
  [DONE] putusan_980_pid.sus_2024_pn_bdg_20260624185133.pdf
     132,858 → 86,241 chars (35.1% reduction)
  [DONE] putusan_9_pid.sus_2026_pn_ngb_20260623014917.pdf
     230,345 → 149,130 chars (35.3% reduction)

 Cleaning complete for 44 file(s)


---

# 7. Save Cleaned Text

Menyimpan setiap teks bersih hasil cleaning ke dalam direktori `/data/raw/` sebagai file `.txt`.

**Aturan penamaan file:** `case_001_<sanitized_original_name>.txt`  
Ini mempertahankan traceability ke file PDF asli sekaligus menambahkan ID berurutan (*sequential ID*).

In [10]:
# Save cleaned text files to /data/raw/
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'raw'

print(" Saving cleaned text files...\n")

saved_count = 0
skipped_count = 0

for i, result in enumerate(extraction_results, 1):
    # Build output filename: case_001_originalname.txt
    original_stem = Path(result['filename']).stem
    safe_name = re.sub(r'[^\w\-]', '_', original_stem)  # sanitize
    output_filename = f"case_{i:03d}_{safe_name}.txt"
    output_path = OUTPUT_DIR / output_filename

    # Store metadata for later use in logging/validation
    result['case_id'] = f"case_{i:03d}"
    result['output_filename'] = output_filename
    result['output_path'] = str(output_path)

    if result.get('cleaned_text'):
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(result['cleaned_text'])
        saved_count += 1
        print(f"  ✅ {output_filename} ({result['cleaned_char_count']:,} chars)")
    else:
        skipped_count += 1
        print(f"  ⏭️  {output_filename} — skipped (empty text)")

print(f"\n📁 Output directory: {OUTPUT_DIR}")
print(f"Saved: {saved_count} file(s)")
if skipped_count > 0:
    print(f"⏭️  Skipped: {skipped_count} file(s)")

 Saving cleaned text files...

  ✅ case_001_putusan_107_pid_sus_2021_pn_jkt_sel_20260624185710.txt (91,347 chars)
  ✅ case_002_putusan_10_pid_sus_2025_pt_bdg_20260624190424.txt (16,671 chars)
  ✅ case_003_putusan_1174_pid_sus_2021_pn_jkt_utr_20260624185123.txt (31,192 chars)
  ✅ case_004_putusan_1221_pid_sus_2020_pt_sby_20260624184940.txt (32,505 chars)
  ✅ case_005_putusan_134_pid_sus_2018_pn_bnr_20260624185857.txt (84,810 chars)
  ✅ case_006_putusan_1357_pid_sus_2021_pt_mdn_20260624190325.txt (85,433 chars)
  ✅ case_007_putusan_1536_pid_sus_2019_pn_jkt_brt_20260624185350.txt (112,755 chars)
  ✅ case_008_putusan_1537_pid_sus_2019_pn_jkt_brt_20260624185405.txt (114,910 chars)
  ✅ case_009_putusan_1597_pid_sus_2019_pn_jkt_utr_20260624185744.txt (53,379 chars)
  ✅ case_010_putusan_159_pid_sus_2023_pn_sda_20260624190433.txt (171,141 chars)
  ✅ case_011_putusan_165_pid_sus_2020_pn_jkt_utr_20260624124027.txt (59,818 chars)
  ✅ case_012_putusan_167_pid_sus_2019_pn_ckr_20260624185039.txt (37,

---

# 8. Logging

Menghasilkan file log terstruktur pada `/logs/cleaning.log` untuk mendokumentasikan seluruh proses preprocessing.

**Detail log untuk setiap file:**
- Nama file asli.
- Status pemrosesan (success/warning/failed).
- Jumlah halaman.
- Jumlah karakter (raw dan cleaned).
- Nama file output.
- Pesan error (jika ada).

In [11]:
# Generate preprocessing log

LOG_PATH = PROJECT_ROOT / 'logs' / 'cleaning.log'
timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

with open(LOG_PATH, 'w', encoding='utf-8') as log_file:
    # Header
    log_file.write(f"{'=' * 72}\n")
    log_file.write(f"  CBR Case Base — Preprocessing Log\n")
    log_file.write(f"  Generated: {timestamp}\n")
    log_file.write(f"  Source folder: {DATASET_DIR}\n")
    log_file.write(f"  Total files: {len(extraction_results)}\n")
    log_file.write(f"{'=' * 72}\n\n")

    # Per-file details
    for result in extraction_results:
        log_file.write(f"File:           {result['filename']}\n")
        log_file.write(f"  Case ID:      {result.get('case_id', 'N/A')}\n")
        log_file.write(f"  Status:       {result['status']}\n")
        log_file.write(f"  Pages:        {result['page_count']}\n")
        log_file.write(f"  Raw chars:    {result.get('raw_char_count', 0):,}\n")
        log_file.write(f"  Cleaned chars:{result.get('cleaned_char_count', 0):,}\n")
        log_file.write(f"  Output file:  {result.get('output_filename', 'N/A')}\n")
        if result.get('error'):
            log_file.write(f"  Error:        {result['error']}\n")
        log_file.write(f"\n")

    # Summary footer
    total = len(extraction_results)
    n_success = sum(1 for r in extraction_results if r['status'] == 'success')
    n_warnings = sum(1 for r in extraction_results if r['status'] == 'warning')
    n_failed = sum(1 for r in extraction_results if r['status'] == 'failed')
    total_pages = sum(r['page_count'] for r in extraction_results)
    total_chars = sum(r.get('cleaned_char_count', 0) for r in extraction_results)

    log_file.write(f"{'=' * 72}\n")
    log_file.write(f"  SUMMARY\n")
    log_file.write(f"{'=' * 72}\n")
    log_file.write(f"  Total files processed:  {total}\n")
    log_file.write(f"  Successful:             {n_success}\n")
    log_file.write(f"  Warnings:               {n_warnings}\n")
    log_file.write(f"  Failed:                 {n_failed}\n")
    log_file.write(f"  Total pages:            {total_pages:,}\n")
    log_file.write(f"  Total cleaned chars:    {total_chars:,}\n")
    log_file.write(f"{'=' * 72}\n")

print(f"Log saved to: {LOG_PATH}")
print(f"\n{'—' * 60}")
print("Log contents:\n")

# Display the log
with open(LOG_PATH, 'r', encoding='utf-8') as f:
    print(f.read())

Log saved to: D:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\logs\cleaning.log

————————————————————————————————————————————————————————————
Log contents:

  CBR Case Base — Preprocessing Log
  Generated: 2026-06-24 21:17:54
  Source folder: D:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\datasets\uu-ite
  Total files: 44

File:           putusan_107_pid.sus_2021_pn_jkt.sel_20260624185710.pdf
  Case ID:      case_001
  Status:       success
  Pages:        41
  Raw chars:    140,425
  Cleaned chars:91,347
  Output file:  case_001_putusan_107_pid_sus_2021_pn_jkt_sel_20260624185710.txt

File:           putusan_10_pid.sus_2025_pt_bdg_20260624190424.pdf
  Case ID:      case_002
  Status:       success
  Pages:        9
  Raw chars:    27,462
  Cleaned chars:16,671
  Output file:  case_002_putusan_10_pid_sus_2025_pt_bdg_20260624190424.txt

File:           putusan_1174_pid.sus_2021_pn_jkt.utr_20260624185123.pdf
  Case ID:      case_003
  Status:    

---

# 9. Validation

Menjalankan quality check otomatis pada hasil extraction:

| Check | Kondisi | Severity |
|-------|-----------|----------|
| Extraction failure | `status == 'failed'` | 🔴 Critical |
| Empty text | `cleaned_char_count == 0` | 🔴 Critical |
| Suspiciously short | `cleaned_char_count < 3000` | 🟡 Warning |

Langkah ini juga menampilkan preview contoh teks dari file pertama yang sukses diekstrak.

In [12]:
# Run validation checks

print(" Running validation checks...\n")

# Threshold: files shorter than this are flagged as suspicious.
# Indonesian court decisions are typically long documents (10+ pages),
# so anything under 3000 chars warrants manual inspection.
MIN_CHAR_THRESHOLD = 3000

issues_critical = []
issues_warning = []

for result in extraction_results:
    filename = result['filename']
    char_count = result.get('cleaned_char_count', 0)

    # Check 1: Extraction failures
    if result['status'] == 'failed':
        issues_critical.append(
            f"❌ FAILED: {filename} — {result.get('error', 'Unknown error')}"
        )

    # Check 2: Empty output
    elif char_count == 0:
        issues_critical.append(
            f"❌ EMPTY: {filename} — No text extracted (0 characters)"
        )

    # Check 3: Suspiciously short text
    elif char_count < MIN_CHAR_THRESHOLD:
        issues_warning.append(
            f"⚠️  SHORT: {filename} — Only {char_count:,} chars "
            f"(threshold: {MIN_CHAR_THRESHOLD:,})"
        )

# Report issues
if issues_critical:
    print("🚨 CRITICAL issues:\n")
    for issue in issues_critical:
        print(f"  {issue}")
    print()

if issues_warning:
    print("⚠️  Warnings:\n")
    for issue in issues_warning:
        print(f"  {issue}")
    print()

if not issues_critical and not issues_warning:
    print(" All files passed validation — no issues detected!\n")

# Check 4: Sisa artefak watermark/disclaimer pada teks yang sudah dibersihkan
remaining_wm = 0
remaining_bleed = 0
for result in extraction_results:
    for line in result.get('cleaned_text', '').split('\n'):
        if is_watermark_line(line):
            remaining_wm += 1
        if is_single_char_bleed(line):
            remaining_bleed += 1

if remaining_wm == 0:
    print("✅ Tidak ada artefak watermark/disclaimer tersisa pada teks yang sudah dibersihkan\n")
else:
    print(f"⚠️  Masih ditemukan {remaining_wm} baris artefak watermark pada teks yang sudah dibersihkan\n")

if remaining_bleed == 0:
    print("✅ Tidak ada baris vertical watermark bleed (satu karakter per baris) tersisa\n")
else:
    print(f"⚠️  Masih ditemukan {remaining_bleed} baris vertical watermark bleed tersisa\n")

# ============================================================
# Sample text preview
# ============================================================
PREVIEW_LENGTH = 1000

print(f"{'—' * 60}")
print("📖 Sample text preview (first successful file):\n")

previewed = False
for result in extraction_results:
    if result.get('cleaned_text'):
        preview = result['cleaned_text'][:PREVIEW_LENGTH]
        print(f"File: {result['filename']}")
        print(f"Case ID: {result.get('case_id', 'N/A')}")
        print(f"{'—' * 40}")
        print(preview)
        print(f"{'—' * 40}")
        remaining = result['cleaned_char_count'] - PREVIEW_LENGTH
        if remaining > 0:
            print(f"(showing first {PREVIEW_LENGTH:,} of {result['cleaned_char_count']:,} chars, "
                  f"{remaining:,} more)")
        previewed = True
        break

if not previewed:
    print("  No text available for preview.")

 Running validation checks...

 All files passed validation — no issues detected!

✅ Tidak ada artefak watermark/disclaimer tersisa pada teks yang sudah dibersihkan

⚠️  Masih ditemukan 20799 baris vertical watermark bleed tersisa

————————————————————————————————————————————————————————————
📖 Sample text preview (first successful file):

File: putusan_107_pid.sus_2021_pn_jkt.sel_20260624185710.pdf
Case ID: case_001
————————————————————————————————————————
g
h
a
m

h
pid.i.a.3
a
rapihkan prin hal terahir gantung p u t u s a n s
nomor 107/pid.sus/2021/pn jkt.sel
demi keadilan berdasarkan ketuhanan yang maha esa
pengadilan negeri jakarta selatan yang mengadili perkara pidana
dengan acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan
sgebagai berikut dalam perkara terdakwa :
1. nama lengkap : topan
2. tempat lahir : makitta
3. umur/tanggal lahir : 36 tahun / 1 desember 1984
h
4. jenis kelamin : laki-laki
a
5. kebangsaan : indonesia
m
6. tempat tinggal : ktp : dusun makitta r

---

# 10. Final Summary

Melihat statistik hasil akhir pipeline dan melakukan penilaian kesiapan data untuk masuk ke Stage 2.

In [13]:
# Final summary statistics

total = len(extraction_results)
success = sum(1 for r in extraction_results if r['status'] == 'success')
warnings_count = sum(1 for r in extraction_results if r['status'] == 'warning')
failed = sum(1 for r in extraction_results if r['status'] == 'failed')
total_pages = sum(r['page_count'] for r in extraction_results)
total_raw_chars = sum(r.get('raw_char_count', 0) for r in extraction_results)
total_cleaned_chars = sum(r.get('cleaned_char_count', 0) for r in extraction_results)

print("=" * 60)
print("STAGE 1 — CASE BASE PREPROCESSING SUMMARY")
print("=" * 60)
print(f"""
  📄 Total PDFs processed:       {total}
  ✅ Successful extractions:      {success}
  ⚠️  Warnings:                   {warnings_count}
  ❌ Failed extractions:          {failed}

  📑 Total pages processed:       {total_pages:,}
  📝 Total raw characters:        {total_raw_chars:,}
  📝 Total cleaned characters:    {total_cleaned_chars:,}

  📁 Cleaned text output:         {PROJECT_ROOT / 'data' / 'raw'}
  📋 Preprocessing log:           {PROJECT_ROOT / 'logs' / 'cleaning.log'}
  📊 Summary CSV:                 {PROJECT_ROOT / 'data' / 'results' / 'preprocessing_summary.csv'}
""")
print("=" * 60)

if failed == 0 and success > 0:
    print("\n Stage 1 complete! The case base is ready for Stage 2 (Case Representation).")
elif failed > 0:
    print(f"\n⚠️  {failed} file(s) failed extraction. Review the log and fix before Stage 2.")
else:
    print("\n No files were successfully processed. Check the dataset path and PDF files.")

STAGE 1 — CASE BASE PREPROCESSING SUMMARY

  📄 Total PDFs processed:       44
  ✅ Successful extractions:      44
  ⚠️  Warnings:                   0
  ❌ Failed extractions:          0

  📑 Total pages processed:       1,458
  📝 Total raw characters:        4,795,600
  📝 Total cleaned characters:    3,061,535

  📁 Cleaned text output:         D:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\data\raw
  📋 Preprocessing log:           D:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\logs\cleaning.log
  📊 Summary CSV:                 D:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\data\results\preprocessing_summary.csv


 Stage 1 complete! The case base is ready for Stage 2 (Case Representation).


In [14]:
# Generate and save a summary DataFrame

summary_df = pd.DataFrame([
    {
        'Case ID': r.get('case_id', f"case_{i:03d}"),
        'Original File': r['filename'],
        'Status': r['status'],
        'Pages': r['page_count'],
        'Raw Chars': r.get('raw_char_count', 0),
        'Cleaned Chars': r.get('cleaned_char_count', 0),
        'Output File': r.get('output_filename', 'N/A'),
        'Error': r.get('error', ''),
    }
    for i, r in enumerate(extraction_results, 1)
])

# Save to CSV
summary_csv_path = PROJECT_ROOT / 'data' / 'results' / 'preprocessing_summary.csv'
summary_df.to_csv(summary_csv_path, index=False, encoding='utf-8')
print(f" Summary saved to: {summary_csv_path}\n")

# Display the table
summary_df

 Summary saved to: D:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\data\results\preprocessing_summary.csv



,Case ID,Original File,Status,Pages,Raw Chars,Cleaned Chars,Output File,Error
0,case_001,putusan_107_pid.sus_2021_pn_jkt.sel_2026062418...,success,41,140425,91347,case_001_putusan_107_pid_sus_2021_pn_jkt_sel_2...,None
1,case_002,putusan_10_pid.sus_2025_pt_bdg_20260624190424.pdf,success,9,27462,16671,case_002_putusan_10_pid_sus_2025_pt_bdg_202606...,None
2,case_003,putusan_1174_pid.sus_2021_pn_jkt.utr_202606241...,success,16,50333,31192,case_003_putusan_1174_pid_sus_2021_pn_jkt_utr_...,None
3,case_004,putusan_1221_pid.sus_2020_pt_sby_2026062418494...,success,20,56034,32505,case_004_putusan_1221_pid_sus_2020_pt_sby_2026...,None
4,case_005,putusan_134_pid.sus_2018_pn_bnr_20260624185857...,success,37,129282,84810,case_005_putusan_134_pid_sus_2018_pn_bnr_20260...,None
5,case_006,putusan_1357_pid.sus_2021_pt_mdn_2026062419032...,success,36,127986,85433,case_006_putusan_1357_pid_sus_2021_pt_mdn_2026...,None
6,case_007,putusan_1536_pid.sus_2019_pn_jkt.brt_202606241...,success,51,170996,112755,case_007_putusan_1536_pid_sus_2019_pn_jkt_brt_...,None
7,case_008,putusan_1537_pid.sus_2019_pn_jkt.brt_202606241...,success,55,177648,114910,case_008_putusan_1537_pid_sus_2019_pn_jkt_brt_...,None
8,case_009,putusan_1597_pid.sus_2019_pn_jkt.utr_202606241...,success,26,84502,53379,case_009_putusan_1597_pid_sus_2019_pn_jkt_utr_...,None
9,case_010,putusan_159_pid.sus_2023_pn_sda_20260624190433...,success,77,263174,171141,case_010_putusan_159_pid_sus_2023_pn_sda_20260...,None
